In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import evaluate
import wandb
from torch.amp import autocast, GradScaler
import nltk
import sys

nltk.download('punkt', quiet=True)

class Config:
    def __init__(self):
        # Resolve paths
        self.base = '.'
        self.excel_path = os.path.join(self.base, '../../data/VideoQuestions.xlsx')
        self.sheet_name = 'Dropped Instances With NER Scor'
        self.clip_path = os.path.join(self.base, '../../embeddings/QCLIP')
        self.eff_path = os.path.join(self.base, '../../embeddings/QEfficient')
        
        # Architecture Settings
        self.alpaca_model = 'MBZUAI/LaMini-GPT-124M' 
        self.t5_model = 'mrm8488/t5-base-finetuned-question-generation-ap'
        self.d_model = 768
        self.num_heads = 8
        
        # Quantum Multi-Stream Settings
        self.n_streams = 8
        self.qubits_per_path = 4
        self.gates_per_path = 15
        
        # Training Parameters
        self.max_input_length = 512
        self.max_target_length = 64
        self.batch_size = 4
        self.grad_accum_steps = 4
        self.num_epochs = 100
        self.lr = 3e-5
        self.weight_decay = 0.01
        self.dropout = 0.1
        self.patience = 10
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.seed = 42

    @property
    def total_qubits(self): return self.n_streams * self.qubits_per_path
    
    @property
    def total_gates(self): return self.n_streams * self.gates_per_path

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f'Device: {cfg.device} | Total Qubits: {cfg.total_qubits}')

In [ ]:
class AdvancedVideoQADataset(Dataset):
    def __init__(self, df, tokenizer, cfg):
        self.samples = []
        print(f'Loading multimodal features from: {cfg.clip_path}')
        for _, row in tqdm(df.iterrows(), total=len(df)):
            v_id, st = row['video_id'], row['start_time']
            c_p = os.path.join(cfg.clip_path, str(v_id), f'{float(st):.1f}', 'embeddings.npy')
            e_p = os.path.join(cfg.eff_path, str(v_id), f'{float(st):.1f}', 'embeddings.npy')
            
            if os.path.exists(c_p) and os.path.exists(e_p):
                c, e = np.load(c_p)[:21], np.load(e_p)[:21]
                if len(c) < 21:
                    c = np.pad(c, ((0, 21-len(c)), (0, 0)))
                    e = np.pad(e, ((0, 21-len(e)), (0, 0)))
                
                input_text = f'Context: {row["chapter title"]} {row["video_title"]} {row["summary"]} Question:'
                inputs = tokenizer(input_text, max_length=cfg.max_input_length, 
                                  padding='max_length', truncation=True, return_tensors='pt')
                
                target_text = str(row['Questions'])
                target = tokenizer(target_text, max_length=cfg.max_target_length, 
                                  padding='max_length', truncation=True, return_tensors='pt')
                
                labels = target['input_ids'].squeeze(0)
                labels[labels == tokenizer.pad_token_id] = -100
                
                self.samples.append({
                    'input_ids': inputs['input_ids'].squeeze(0),
                    'attention_mask': inputs['attention_mask'].squeeze(0),
                    'clip': torch.from_numpy(c).float(),
                    'eff': torch.from_numpy(e).float(),
                    'labels': labels
                })
        
        if not self.samples:
            print('WARNING: Zero valid samples found. Please check your data paths.')
        else:
            print(f'Dataset ready: {len(self.samples)} valid samples.')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

tokenizer = AutoTokenizer.from_pretrained(cfg.alpaca_model)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

df = pd.read_excel(cfg.excel_path, sheet_name=cfg.sheet_name)
df = df[df['Questions'].notnull()]
train_df, val_df = train_test_split(df, test_size=0.1, random_state=cfg.seed)

train_loader = DataLoader(AdvancedVideoQADataset(train_df, tokenizer, cfg), 
                          batch_size=cfg.batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(AdvancedVideoQADataset(val_df, tokenizer, cfg), 
                        batch_size=cfg.batch_size, pin_memory=True)

In [ ]:
def create_quantum_path(n_qubits, n_gates):
    dev = qml.device('lightning.qubit', wires=n_qubits)
    @qml.qnode(dev, interface='torch')
    def circuit(inputs):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
        param_idx = 0
        layer = 0
        while param_idx < n_gates:
            for i in range(n_qubits):
                if param_idx < n_gates:
                    qml.RY(inputs[..., param_idx], wires=i)
                    param_idx += 1
                if param_idx < n_gates:
                    qml.RZ(inputs[..., param_idx], wires=i)
                    param_idx += 1
            if n_qubits > 1:
                for i in range(n_qubits):
                    if layer % 2 == 0: qml.CNOT(wires=[i, (i + 1) % n_qubits])
                    else: qml.CNOT(wires=[(i + 1) % n_qubits, i])
            layer += 1
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    return circuit

class SOTAQuantumVideoAlpaca(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.alpaca = AutoModelForCausalLM.from_pretrained(cfg.alpaca_model)
        self.clip_proj = nn.Linear(768, cfg.d_model)
        self.eff_proj = nn.Linear(1792, cfg.d_model)
        self.cross_attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, batch_first=True)
        self.ln_fusion = nn.LayerNorm(cfg.d_model)
        self.q_down = nn.Linear(cfg.d_model, cfg.total_gates)
        self.q_paths = nn.ModuleList([
            qml.qnn.TorchLayer(create_quantum_path(cfg.qubits_per_path, cfg.gates_per_path), weight_shapes={})
            for _ in range(cfg.n_streams)
        ])
        self.q_up = nn.Linear(cfg.total_qubits, cfg.d_model)
        self.ln_q = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.multimodal_cross_attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, batch_first=True)
        self.ln_multimodal = nn.LayerNorm(cfg.d_model)
        
    def get_video_context(self, clip_feats, eff_feats):
        batch, seq, _ = clip_feats.shape
        c, e = self.clip_proj(clip_feats), self.eff_proj(eff_feats)
        fused, _ = self.cross_attn(query=e, key=c, value=c)
        fused = self.ln_fusion(fused + e) 
        q_params = self.q_down(fused).reshape(-1, self.cfg.total_gates)
        q_outputs = []
        for i in range(self.cfg.n_streams):
            start = i * self.cfg.gates_per_path
            end = (i + 1) * self.cfg.gates_per_path
            q_outputs.append(self.q_paths[i](q_params[:, start:end]))
        q_combined = torch.cat(q_outputs, dim=-1).reshape(batch, seq, self.cfg.total_qubits)
        return self.ln_q(fused + self.dropout(self.q_up(q_combined)))

    def forward(self, clip_feats, eff_feats, input_ids, attention_mask, labels=None):
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        inputs_embeds = self.alpaca.get_input_embeddings()(input_ids)
        
        mm_fused, _ = self.multimodal_cross_attn(query=inputs_embeds, key=video_ctx, value=video_ctx)
        final_embeds = self.ln_multimodal(inputs_embeds + self.dropout(mm_fused))
        
        outputs = self.alpaca(inputs_embeds=final_embeds, attention_mask=attention_mask, 
                              labels=labels, output_hidden_states=True, return_dict=True)
        # For contrastive loss alignment, we use the last hidden state as 'encoder output'
        outputs.encoder_last_hidden_state = outputs.hidden_states[-1]
        return outputs

    def generate(self, clip_feats, eff_feats, input_ids, attention_mask, **kwargs):
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        inputs_embeds = self.alpaca.get_input_embeddings()(input_ids)
        mm_fused, _ = self.multimodal_cross_attn(query=inputs_embeds, key=video_ctx, value=video_ctx)
        final_embeds = self.ln_multimodal(inputs_embeds + self.dropout(mm_fused))
        
        return self.alpaca.generate(inputs_embeds=final_embeds, attention_mask=attention_mask, **kwargs)

model = SOTAQuantumVideoAlpaca(cfg).to(cfg.device)

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2):
        tensor1 = output1.encoder_last_hidden_state if hasattr(output1, 'encoder_last_hidden_state') else output1[0]
        tensor2 = output2.encoder_last_hidden_state if hasattr(output2, 'encoder_last_hidden_state') else output2[0]

        if tensor1.size(1) != tensor2.size(1):
            max_seq = max(tensor1.size(1), tensor2.size(1))
            tensor1 = F.pad(tensor1, (0, 0, 0, max_seq - tensor1.size(1)))
            tensor2 = F.pad(tensor2, (0, 0, 0, max_seq - tensor2.size(1)))
        
        if tensor1.size(2) != tensor2.size(2):
            max_hid = max(tensor1.size(2), tensor2.size(2))
            tensor1 = F.pad(tensor1, (0, max_hid - tensor1.size(2)))
            tensor2 = F.pad(tensor2, (0, max_hid - tensor2.size(2)))
        
        return F.mse_loss(tensor1, tensor2)

model_2 = AutoModelForSeq2SeqLM.from_pretrained(cfg.t5_model).to(cfg.device)
tokenizer_2 = AutoTokenizer.from_pretrained(cfg.t5_model)
contrastive_loss_fn = ContrastiveLoss()

In [ ]:
rouge = evaluate.load('rouge', quiet=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler('cuda')
best_rouge_l = 0

wandb.init(project='VQG-Quantum-Contrastive', name='Quantum-Alpaca-Contrastive')

for epoch in range(cfg.num_epochs):
    model.train()
    total_train_loss, train_steps = 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
    for i, batch in enumerate(pbar):
        c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
        in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
        
        with autocast('cuda'):
            outputs = model(c, e, in_ids, attn_mask, l)
            loss_ce = outputs.loss
            
            # Teacher Alignment logic
            inputs_2_str = tokenizer.batch_decode(in_ids, skip_special_tokens=True)
            encoded_inputs_2 = tokenizer_2(inputs_2_str, max_length=cfg.max_input_length, 
                                          truncation=True, padding=True, return_tensors='pt').to(cfg.device)
            with torch.no_grad():
                model_2_outputs = model_2(input_ids=encoded_inputs_2['input_ids'], 
                                         attention_mask=encoded_inputs_2['attention_mask'],
                                         decoder_input_ids=encoded_inputs_2['input_ids'])
            
            loss_contrastive = contrastive_loss_fn(outputs, model_2_outputs)
            loss = (loss_ce + loss_contrastive) / max(1, cfg.grad_accum_steps)
            
        scaler.scale(loss).backward()
        if (i + 1) % cfg.grad_accum_steps == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            
        total_train_loss += loss.item() * cfg.grad_accum_steps
        train_steps += 1
        pbar.set_postfix({'loss': total_train_loss / max(1, train_steps)})
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
            in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
            
            gen_ids = model.generate(c, e, in_ids, attn_mask, max_length=cfg.max_target_length, num_beams=5)
            all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
            all_labels.extend([tokenizer.decode(g[g != -100], skip_special_tokens=True) for g in l])

    r_res = rouge.compute(predictions=all_preds, references=all_labels)
    print(f'Epoch {epoch+1} | Loss: {total_train_loss/train_steps:.4f} | RougeL: {r_res["rougeL"]:.4f}')
    wandb.log({'epoch': epoch+1, 'train_loss': total_train_loss/train_steps, 'rougeL': r_res['rougeL']})
    
    if r_res['rougeL'] > best_rouge_l:
        best_rouge_l = r_res['rougeL']
        torch.save(model.state_dict(), 'QuantumAlpaca_RTheta.pt')

wandb.finish()

In [ ]:
print('Loading best model for final testing...')
model.load_state_dict(torch.load('QuantumAlpaca_RTheta.pt'))
model.eval()

rouge = evaluate.load('rouge', quiet=True)
bleu = evaluate.load('bleu', quiet=True)
meteor = evaluate.load('meteor', quiet=True)
bertscore = evaluate.load('bertscore', quiet=True)
nltk.download('wordnet', quiet=True)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Testing'):
        c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
        in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
        
        gen_ids = model.generate(c, e, in_ids, attn_mask, max_length=cfg.max_target_length, num_beams=5)
        all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
        all_labels.extend([tokenizer.decode(g[g != -100], skip_special_tokens=True) for g in l])

final_metrics = {
    'rougeL': rouge.compute(predictions=all_preds, references=all_labels)['rougeL'],
    'bleu': bleu.compute(predictions=all_preds, references=[[l] for l in all_labels])['bleu'],
    'meteor': meteor.compute(predictions=all_preds, references=all_labels)['meteor']
}
print('\nFinal Test Metrics:')
print(json.dumps(final_metrics, indent=4))